# 🐢 Tortuga — Re-treino YOLOv8 (Fase 1) no Google Colab — versão CORRIGIDA

**Challenge SPI 2026 · FIAP × SPI**

Este notebook treina o detector de EPIs **sem repetir o erro que quebrou o modelo anterior**.

### O que deu errado antes
O re-treino de 29 mil imagens misturou datasets que rotulavam **só parte** dos EPIs. Resultado:
protetor auricular ficou com **0 imagens** e o modelo aprendeu que óculos/botas não rotulados eram
"fundo" — degradando tudo. (Diagnóstico completo em `CORRECOES_TORTUGA.md`.)

### O que muda aqui
- **Uma única base consistente: SH17** (8.099 imagens, anotação completa por imagem). Inclui
  **protetor auricular (ear-muffs), capacete, óculos, colete e calçado** — então **não há
  contaminação por anotação parcial**. SH17 é citado como fonte aceita no próprio descritivo.
- **Checagem pré-voo**: conta as instâncias por classe e **aborta** se alguma ficar em 0
  (impede queimar horas de GPU num dataset quebrado).
- **Validação com métricas por classe** e instruções claras de implantação no app.

### Setup de GPU recomendado (você tem ~70 compute units)
| GPU (runtime) | VRAM | ~CU/h | Tempo (SH17, 100 ép.) | Custo aprox. | Quando usar |
|---|---|---|---|---|---|
| **A100 (High-RAM)** | 40 GB | ~15 | **~1,5–2,5 h** | ~25–35 CU | **Recomendado p/ o run final** |
| **L4** | 22,5 GB | ~4–5 | ~3–5 h | ~15–25 CU | Melhor custo p/ iterar |
| T4 (grátis) | 15 GB | ~1,8 | ~6–9 h | ~12–16 CU | Só testes rápidos |

**Recomendação:** **A100 + High-RAM** para o run final (sobra folga nos 70 CU para uma 2ª rodada).
Use **L4** se quiser iterar barato. Defina em **Runtime → Change runtime type → A100 GPU + High-RAM**.


In [ ]:
# ── Célula 1: GPU, VRAM e RAM disponíveis ─────────────────────
import subprocess, torch
g = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                   capture_output=True, text=True).stdout.strip()
print('GPU :', g or 'NENHUMA — selecione uma GPU em Runtime > Change runtime type')
with open('/proc/meminfo') as f:
    ram = int(next(l for l in f if 'MemTotal' in l).split()[1]) / 1024 / 1024
print(f'RAM : {ram:.1f} GB')
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory/1e9
    print(f'VRAM: {vram:.1f} GB')


In [ ]:
# ── Célula 2: Dependências ────────────────────────────────────
!pip -q install ultralytics roboflow
import ultralytics; ultralytics.checks()


In [ ]:
# ── Célula 3: Google Drive (salva o modelo treinado) ──────────
from google.colab import drive
import os
drive.mount('/content/drive')
DRIVE_OUT = '/content/drive/MyDrive/Tortuga/models'
os.makedirs(DRIVE_OUT, exist_ok=True)
print('Modelos finais irão para:', DRIVE_OUT)


In [ ]:
# ── Célula 4: Configuração ────────────────────────────────────
# Cole sua chave do Roboflow (Settings > API). NÃO comite a chave no Git.
ROBOFLOW_API_KEY = 'COLE_SUA_CHAVE_AQUI'

import os
os.environ['ROBOFLOW_API_KEY'] = ROBOFLOW_API_KEY
DATA_DIR  = '/content/datasets'
MERGED    = '/content/datasets/tortuga_clean'
os.makedirs(DATA_DIR, exist_ok=True)

# Parâmetros de treino
MODEL_SIZE = 'yolov8s.pt'   # 's' = bom equilíbrio; use 'yolov8m.pt' na A100 p/ +precisão
IMGSZ      = 640            # igual ao app (gui.py roda inferência em 640)
EPOCHS     = 100
PATIENCE   = 20
print('Config OK | modelo base:', MODEL_SIZE)


## Estratégia de dataset (por que SH17 sozinho)

Misturar fontes que anotam EPIs diferentes **reintroduz a contaminação**. Por isso a base é
**uma só** (SH17), onde **todos** os EPIs de cada imagem estão rotulados. Mapeamos as classes do
SH17 para os nomes do Tortuga e **descartamos** classes irrelevantes (partes do corpo, ferramentas).
Descartar caixas de "head/hands/foot" não contamina nada — as caixas de EPI continuam corretas.

> Quer detecção de **violação** (NO-Capacete etc.) e robustez de sala? A forma certa é **adicionar
> suas próprias imagens da sala da FIAP** rotuladas (exigência do item 4.6) — veja a célula opcional.


In [ ]:
# ── Célula 5: Download do SH17 (Roboflow) ─────────────────────
# Alternativas se este slug mudar:
#  - Kaggle:  mugheesahmad/sh17-dataset-for-ppe-detection
#  - GitHub:  ahmadmughees/SH17dataset
from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
# Ajuste workspace/project/version conforme a página do dataset no Roboflow Universe.
project = rf.workspace('ppehaak').project('sh17-hmkpl')
version = project.version(2)
ds = version.download('yolov8', location=f'{DATA_DIR}/sh17')
print('SH17 baixado em:', ds.location)


In [ ]:
# ── Célula 6: Remapeia a fonte -> taxonomia Tortuga (por PALAVRA-CHAVE) ──
# Robusto a variações de nome (ear-mufs/ear-muffs, hard-hat/helmet, shoe/boots...).
# Imprime as classes REAIS da fonte para nada ser adivinhado.
import yaml, shutil, collections
from pathlib import Path

TARGET = ['Hardhat','Safety Vest','Safety Glasses','Mask',
          'Ear Protection','Safety Boots','Gloves','Person']
tgt_id = {c:i for i,c in enumerate(TARGET)}

def map_class(name):
    n = name.lower().strip().replace('_','-').replace(' ','-')
    # protetor auricular: muff/plug/ear+protect — NUNCA 'ear' sozinho (orelha)
    if 'muf' in n or 'plug' in n or ('ear' in n and 'protect' in n): return 'Ear Protection'
    if 'helmet' in n or 'hardhat' in n or 'hard-hat' in n:           return 'Hardhat'
    if 'vest' in n:                                                  return 'Safety Vest'
    if 'glass' in n or 'goggle' in n:                                return 'Safety Glasses'
    if 'mask' in n:                                                  return 'Mask'
    if 'boot' in n or 'shoe' in n:                                   return 'Safety Boots'
    if 'glove' in n:                                                 return 'Gloves'
    if n == 'person' or 'worker' in n:                              return 'Person'
    return None   # ear, face, face-guard, foot, hands, head, suit, tool... -> descartada

src = Path(ds.location)
src_names = [str(x) for x in yaml.safe_load(open(src/'data.yaml'))['names']]

src_counts = collections.Counter()
for sp in ['train','valid','test']:
    ld = src/sp/'labels'
    if ld.exists():
        for t in ld.glob('*.txt'):
            for line in open(t):
                p = line.split()
                if p and int(p[0]) < len(src_names):
                    src_counts[src_names[int(p[0])]] += 1
print('FONTE: classe -> alvo Tortuga | instancias')
for nm in src_names:
    print(f'  {nm:26} -> {str(map_class(nm) or "DESCARTADA"):16} | {src_counts[nm]}')

out = Path(MERGED)
if out.exists(): shutil.rmtree(out)
counts = {c:0 for c in TARGET}
for sp in ['train','valid','test']:
    (out/sp/'images').mkdir(parents=True, exist_ok=True)
    (out/sp/'labels').mkdir(parents=True, exist_ok=True)
    idir, ldir = src/sp/'images', src/sp/'labels'
    if not idir.exists(): continue
    for img in idir.glob('*.*'):
        lbl = ldir/(img.stem+'.txt')
        if not lbl.exists(): continue
        new = []
        for line in open(lbl):
            p = line.split()
            if not p: continue
            i = int(p[0])
            if i >= len(src_names): continue
            tg = map_class(src_names[i])
            if tg is None: continue
            new.append(f"{tgt_id[tg]} " + " ".join(p[1:]))
            if sp == 'train': counts[tg] += 1
        if not new: continue
        shutil.copy2(img, out/sp/'images'/img.name)
        open(out/sp/'labels'/(img.stem+'.txt'),'w').write("\n".join(new))

yaml.safe_dump({'path':str(out),'train':'train/images','val':'valid/images',
                'test':'test/images','nc':len(TARGET),'names':TARGET},
               open(out/'data.yaml','w'), allow_unicode=True)
print('\nDataset limpo em:', out)


In [ ]:
# ── Célula 7: CHECAGEM PRÉ-VOO (impede dataset quebrado) ──────
MIN_INST = 150
total = sum(counts.values()) or 1
print('Instâncias por classe (treino):')
zeros = []
for c in TARGET:
    n = counts[c]; flag = ''
    if n == 0: flag = '  <<< ZERO — NÃO TREINE assim!'; zeros.append(c)
    elif n < MIN_INST: flag = f'  <<< poucas (< {MIN_INST})'
    print(f'  {c:16} {n:>6}  ({100*n/total:4.1f}%){flag}')
assert not zeros, f'Classes vazias: {zeros}. Conserte a fonte antes de treinar.'
print('\nOK — nenhuma classe vazia. Pode treinar.')


## (Opcional, MUITO recomendado) Adicione imagens da sua sala

O descritivo (item 4.6) **exige** dados próprios do grupo. Coletar 100–300 frames da sala da FIAP
(pessoas com/sem cada EPI), rotular no Roboflow/CVAT com **as mesmas 8 classes acima** e colocá-los
em `/content/datasets/tortuga_clean/{train,valid}/...` antes de treinar:
1. fecha o **gap de domínio** (sala ≠ canteiro de obra) — principal causa de "não pega na sala";
2. permite rotular **NO-Capacete/NO-Colete** etc. se você quiser detecção de **violação**.


In [ ]:
# ── Célula 8: Treino ──────────────────────────────────────────
from ultralytics import YOLO
import torch

# batch automático conforme a VRAM
vram = torch.cuda.get_device_properties(0).total_memory/1e9 if torch.cuda.is_available() else 0
batch = 48 if vram>=38 else 24 if vram>=20 else 12 if vram>=14 else 8
print(f'VRAM {vram:.0f} GB -> batch {batch}')

model = YOLO(MODEL_SIZE)
results = model.train(
    data=f'{MERGED}/data.yaml',
    epochs=EPOCHS, imgsz=IMGSZ, batch=batch, device=0,
    project='/content/runs', name='tortuga_sh17', exist_ok=True,
    patience=PATIENCE, cache=True, plots=True,
    # augmentation moderada -> generaliza p/ sala/escritório
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, degrees=8, translate=0.1, scale=0.5,
    fliplr=0.5, mosaic=1.0, mixup=0.1, erasing=0.3,
)
print('Treino concluído.')


In [ ]:
# ── Célula 9: Validação + métricas por classe ─────────────────
best = '/content/runs/tortuga_sh17/weights/best.pt'
m = YOLO(best)
metrics = m.val(data=f'{MERGED}/data.yaml', split='test')
print(f"\nGERAL  mAP@0.5={metrics.box.map50:.3f} (meta >=0.75) | "
      f"mAP@0.5:0.95={metrics.box.map:.3f} | P={metrics.box.mp:.3f} | R={metrics.box.mr:.3f}")
print('\nPor classe (mAP@0.5):')
for i, c in enumerate(TARGET):
    try: print(f'  {c:16} {metrics.box.maps[i]:.3f}')
    except Exception: pass


In [ ]:
# ── Célula 10: Exporta e salva no Drive ───────────────────────
import shutil, datetime
stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M')
dst = f'{DRIVE_OUT}/best_sh17_{stamp}.pt'
shutil.copy2(best, dst)
print('Modelo salvo no Drive:', dst)
print('Tamanho:', round(__import__('os').path.getsize(best)/1e6,1), 'MB')


## Como implantar no app Tortuga

1. Baixe `best_sh17_*.pt` do Drive para `server/models/` no seu PC e renomeie para **`best_retrained.pt`**.
2. Teste ao vivo antes de promover:
   ```bash
   cd server
   python gui.py --model models/best_retrained.pt --det-size 960
   ```
   Confirme que **capacete, colete, óculos, protetor auricular e bota/calçado** aparecem na sala.
3. Só se detectar bem, promova para produção:
   ```bash
   copy models\best_retrained.pt models\best.pt   # Windows
   ```
   Como este modelo tem >4 classes, o `detector.py` entra automaticamente no **modo SINGLE** e usa
   só ele (deixa de usar o par capacete+óculos restaurado).

### Observações honestas
- **Calçado ≠ bota de segurança**: o SH17 rotula "shoes" (calçado em geral). Para a banca, declare isso
  ou rotule alguns frames de bota real (próprios) para refinar.
- **Detecção × violação**: este modelo detecta a *presença* dos EPIs (caixas verdes). Para acusar
  *ausência* (NO-Capacete), rotule também esses casos nos seus dados próprios — é o caminho correto
  e atende ao item 4.6.
- Rode **`--validate`** e registre as métricas por classe para a arguição (a banca valoriza dados, não opiniões).
